In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import holidays 
import sys
sys.path.append("..") 
from src.features.feature_sets import get_feature_sets, TARGET

In [ ]:
df = pd.read_parquet(r'C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data/processed/tabla_maestra_procesada.parquet')

In [ ]:
df.isna().sum().sort_values(ascending=False).head(30)

In [ ]:
def racha_maxima_nan(s):
    mask =s.isna()
    ids= (~mask).cumsum()
    return mask.groupby(ids).sum().max()



In [ ]:
rachas = df.apply(racha_maxima_nan)
rachas[rachas > 0].value_counts().sort_index()



In [ ]:
def imputar_capa1(df, columnas=None):
    df = df.copy()
    if columnas is None:
        columnas = df.select_dtypes(include='number').columns.drop('precio_espana')
    df = df.set_index("datetime_utc")
    df[columnas] = df[columnas].interpolate(method='time', limit=6, limit_area='inside')
    df = df.reset_index()
    return df

In [ ]:
df_copia=imputar_capa1(df)

In [ ]:
antes = df.isna().sum().sum()
despues = df_copia.isna().sum().sum()
print(antes, despues)

## Imputación de NaN — Capa 1: interpolación temporal

**Estrategia general.** La imputación se aborda en dos capas. La capa 1 cubre los
huecos cortos mediante interpolación temporal determinista; los huecos largos se
derivan a la capa 2 (donantes AEMET) o se excluyen por completitud (ESIOS).

**Método y parámetros** (`imputar_capa1`):
- `method="time"`: interpola según la distancia temporal real entre horas, no por
  posición de fila. Imprescindible por los saltos de índice del cambio de hora (DST).
- `limit=6`: frontera capa1/capa2. Se reconstruyen huecos de ≤6h consecutivas; los
  más largos se dejan intactos para su tratamiento específico. Es una regla permanente
  del pipeline, no ajustada al histórico actual (blinda micro-fallos futuros en producción).
- `limit_area="inside"`: solo rellena huecos con dato válido a ambos lados. Protege la
  cola estructural (`2026-06-30`) de cualquier extrapolación hacia el futuro.

**Selección de columnas.** Regla estructural (no manual): todas las numéricas
(`select_dtypes`), excluyendo el target `precio_espana`. El target nunca se interpola:
si falta el label, la fila se excluye del entrenamiento, no se fabrica un valor.

**Resultado verificado.** NaN totales: 42.853 → 38.377 (−4.476 huecos cortos)

In [ ]:
df_tmed = df_copia.loc[:,df.columns.str.startswith('tmed_')]
df_tmed.corr()


In [ ]:
nulos = df_copia.isna().sum().sort_values(ascending = False)
nulos[nulos > 0].sort_values(ascending = False)

racha = (racha_maxima_nan, nulos)
rachas = rachas[rachas > 0].sort_index()
mascara = df_copia['tmed_medina_de_pomar'].isna()
resultado= df_copia.loc[mascara, 'tmed_san_pedro_manrique'].isna().sum()
mascara = df_copia['tmed_medina_de_pomar'].isna()
resultado_2= df_copia.loc[mascara, 'tmed_medina_de_pomar'].isna().sum()
df_copia['tmed_medina_de_pomar'].isna().sum()

# rachas['tmed_san_pedro_manrique']



In [ ]:


x = df_copia['tmed_san_pedro_manrique']   
y = df_copia['tmed_medina_de_pomar']      

plt.figure(figsize=(6, 6))
plt.scatter(x, y, s=5, alpha=0.2)        
plt.plot([-5, 40], [-5, 40], 'r--')      
plt.xlabel('San Pedro Manrique (°C)')
plt.ylabel('Medina de Pomar (°C)')
plt.show()

In [ ]:
resta = df_copia['tmed_medina_de_pomar']-df_copia['tmed_san_pedro_manrique']
media = resta.mean()
media

In [ ]:
meses = df_copia['datetime_utc'].dt.month
offset_mensual = resta.groupby(df_copia['datetime_utc'].dt.month).mean()
offset_fila = meses.map(offset_mensual)
offset_fila 

donante = df_copia['tmed_san_pedro_manrique'] + offset_fila

df_copia['tmed_medina_de_pomar'] = df_copia['tmed_medina_de_pomar'].fillna(donante)

In [ ]:
candidatas = df_tmed
df_tmed = df_copia.columns[df_copia.columns.str.startswith("tmed_")]
def imputar_donante(df, receptora, candidatas, umbral = 0.8):
    df = df.copy()
    candidatas = candidatas.drop(receptora)
    ranking = df[candidatas].corrwith(df[receptora]).sort_values(ascending = False)
    donante = None
    mascara_receptora = df[receptora].isna()
    n_huecos = mascara_receptora.sum()
    for candidato in ranking.index:
        coinciden = df.loc[mascara_receptora, candidato].isna().sum()
        cobertura = 1 - coinciden / n_huecos
        if cobertura > umbral :
            donante = candidato
            break
    if donante is None:
        return df
    else:
        meses = df['datetime_utc'].dt.month
        resta = df[receptora]-df[donante]
        offset_mensual = resta.groupby(meses).mean()
        offset_fila = meses.map(offset_mensual)
        donante_ajustado = df[donante] + offset_fila
        df[receptora] = df[receptora].fillna(donante_ajustado)
        return df

def imputar_todas_donante(df, variables, umbral=0.8):
    for var in variables:
        columnas_var = df.columns[df.columns.str.startswith(var)]
        for receptora in columnas_var:
            if df[receptora].isna().sum() > 0:
                df = imputar_donante(df, receptora, columnas_var, umbral)
    return df
        
# df_copia = imputar_donante(df_copia, "tmed_medina_de_pomar", df_tmed.columns)
# df_copia['tmed_medina_de_pomar'].isna().sum()

 

In [ ]:
nulos_temperatura=df_copia.columns[df_copia.columns.str.startswith('tmed_')]
df_copia[nulos_temperatura].isna().sum()

In [ ]:

df_copia = imputar_capa1(df)                                  
df_copia = imputar_todas_donante(df_copia, ["tmed_", "tmax_"]) 

columnas_temp = df_copia.columns[df_copia.columns.str.startswith(("tmed_", "tmax_"))]
df_copia[columnas_temp] = df_copia[columnas_temp].ffill().bfill()  

df_copia[columnas_temp].isna().sum().sum()   

## Imputación de NaN — Capa 2: donantes entre estaciones AEMET

**Problema.** Los bloques largos de AEMET (estaciones caídas semanas: Medina de Pomar
~43 días, Miranda ~87 días) no los puede tapar la interpolación temporal. Se reconstruyen
copiando la señal de otra estación "donante" con clima parecido.

**Selección de donante (dos criterios, en orden):**
1. **Correlación** (`corrwith`): se ordenan las estaciones candidatas de mayor a menor
   correlación con la receptora.
2. **Cobertura** (puerta del 80%): se baja por ese ranking y se elige la primera candidata
   que tenga dato en ≥80% de los huecos de la receptora. La correlación sola no basta: una
   estación vecina puede haberse caído a la vez (co-fallo) — máxima correlación pero cobertura
   nula. El umbral separa "vecino que co-falló" de "donante bueno".

**Ajuste de nivel (offset mensual).** El donante no es la receptora (diferencia de altitud →
sesgo sistemático). Se corrige con un offset aditivo calculado por mes:
`offset_mes = media(receptora − donante)` sobre el solape. Se eligió mensual (no único) porque
el offset varía con la estación del año (verificado: rango ~1,8–3,1 °C). El relleno es
`donante + offset_del_mes`, y solo sobre los NaN de la receptora (`fillna`).

**Residual.** Un donante único no cubre el 100%: donde el donante también falla, se deja NaN.
Regla: un donante por estación; lo no cubierto se tapa con un barrido final `ffill().bfill()`
acotado a las columnas de temperatura.

**Alcance.** Validado y aplicado a temperatura (`tmed`, `tmax`), donde las correlaciones entre
estaciones son 0,9+. NO se aplica a `prec` (precipitación local, correlaciones 0,3–0,6, y el
offset aditivo podría dar valores negativos) ni todavía a viento/sol (pendiente de revisar
correlación). `tmin` pendiente de re-ingesta desde AEMET.

**Automatización.** Función reutilizable `imputar_donante(df, receptora, candidatas, umbral)`
+ orquestadora `imputar_todas_donante(df, variables)` que recorre cada variable y estación.
Determinista, sin nombres hardcodeados → lista para `src/features/` y Prefect.

**Lección (MLOps).** `imputar_donante` mutaba `df` en el sitio, lo que corrompía el estado al
ejecutar celdas fuera de orden. Se corrigió con `df = df.copy()` al inicio: toda función que
transforma datos debe trabajar sobre una copia y devolver una nueva, nunca mutar su entrada.

**Resultado.** Temperatura completamente imputada (0 NaN tras el barrido), de forma reproducible.





In [ ]:
velmedia = df_copia.loc[:, df_copia.columns.str.startswith("velmedia_")]
velmedia.corr()
velmedia = df_copia.loc[:, df_copia.columns.str.startswith("prec")]
velmedia.corr()

In [ ]:
def imputar_climatologia (df, columnas):
    df = df.copy()
    meses = df['datetime_utc'].dt.month 
    for columna in columnas:
        media =df[columna].groupby(meses).mean()
        relleno = meses.map(media)
        df[columna] = df[columna].fillna(relleno)
    return df
    


In [ ]:
columnas = df.columns[df.columns.str.startswith(('vel','racha_', 'prec', 'sol'))]
df_copia= imputar_climatologia(df_copia, columnas)

df_copia[columnas].isna().sum().sum()





## Paso 4 — Marca de completitud (versión general)

Los NaN que quedan (día de 23h de ESIOS + cola `2026-06-30`) **no se imputan**: se **marcan**.
Una fila es *entrenable* si tiene todas sus features válidas y el target. Es general por ahora;
se refinará por modelo (LSTM / XGBoost) cuando se definan sus sets de features.

# Fase 3 · Feature Engineering — Bloque 1: Imputación de NaN y marca de completitud

## Objetivo
Tratar los NaN que el ETL dejó a propósito sin tocar, con una estrategia **determinista**
(interpolación, medias, climatología) para que el pipeline nunca se rompa y sea reproducible
en producción (re-ejecución semanal con Prefect). Nada de métodos que dependan de un objeto
fiteado o de aleatoriedad.

## Principio rector
La estrategia de imputación se decide por la **CAUSA** del hueco, no por la columna. Tres familias:
- **Micro-huecos** (fallos puntuales de registro) → interpolación temporal.
- **Bloques largos** (estación AEMET caída semanas) → donante entre estaciones.
- **Cola estructural** (OMIE llega más lejos que ESIOS) → no se imputa; se marca.

## Diagnóstico de rachas
`.isna().sum()` cuenta NaN totales, pero no distingue "100 huecos de 1h" de "1 bloque de 100h",
que piden tratamientos opuestos. Para verlo se cuenta la **racha máxima de NaN consecutivos**
por columna con el truco de máscara + `(~mask).cumsum()` (la suma acumulada se "congela"
durante los NaN → cada bloque comparte etiqueta) + `groupby().sum().max()`.
Hallazgo: no había micro-goteo real; todo hueco era de ≥23h. La distribución era un continuo
(24h → 2088h) con un único corte natural tras las 24h.

## Capa 1 — Interpolación temporal
Función `imputar_capa1(df, columnas=None)`.
- `method="time"`: interpola por distancia temporal real (clave por el DST), no por posición de fila.
- `limit=6`: frontera capa1/capa2. Regla permanente del pipeline, no ajustada al histórico:
  reconstruye micro-fallos ≤6h; lo más largo lo deja para otra capa.
- `limit_area="inside"`: solo huecos con dato a ambos lados → blinda la cola de la extrapolación.
- Selección de columnas por regla (`select_dtypes("number")` menos el target), no a mano.
- El target `precio_espana` NUNCA se interpola: fabricar un label es peor que excluir la fila.
Resultado: 42.853 → 38.377 NaN.

## Capa 2 — Donantes entre estaciones AEMET (solo temperatura)
Reconstruir la estación rota copiando la de una **donante** con clima parecido.
Selección de donante con DOS criterios en orden:
1. **Correlación** (`corrwith`) — ordena candidatas.
2. **Cobertura ≥80%** — baja por el ranking hasta la primera con dato en el hueco.
La correlación sola


In [ ]:
local = df_copia['datetime_utc'].dt.tz_convert('Europe/Madrid')
df_copia['hora']= local.dt.hour
df_copia['dia']= local.dt.day
df_copia['semana']= local.dt.dayofweek
df_copia['mes']= local.dt.month




In [ ]:
# Calendario de festivos NACIONALES de España para los años de tu histórico
festivos_es = holidays.Spain(years=range(2022, 2027))

# Marca cada fila según su fecha LOCAL
df_copia['festivo'] = local.dt.date.isin(set(festivos_es)).astype(int)

In [ ]:
# Seno y coseno de dia, semana y mes
ciclos = {"hora": 24, "semana": 7, "mes": 12}
for col, periodo in ciclos.items():
    df_copia[f"{col}_sin"] = np.sin(2 * np.pi * df_copia[col] / periodo)
    df_copia[f"{col}_cos"] = np.cos(2 * np.pi * df_copia[col] / periodo)

# comprobacion 
# df_copia[['hora_sin','hora_cos']].describe()  rango entre -1 y 1


In [ ]:
precio = df_copia.set_index('datetime_utc')['precio_espana']   # tabla: fecha -> precio

df_copia['precio_lag_24']  = (df_copia['datetime_utc'] - pd.Timedelta(hours=24)).map(precio)
df_copia['precio_lag_168'] = (df_copia['datetime_utc'] - pd.Timedelta(hours=168)).map(precio)

# media precio movil (para suavizar lags puntuales)
media_24 = precio.rolling('24h').mean()
df_copia['precio_media_24'] = (df_copia['datetime_utc'] - pd.Timedelta(hours=24)).map(media_24)

In [ ]:
def auditar_imputacion(df_crudo, df_imp, columnas):
    """Observado (gris) vs imputado (rojo) para cada columna que tuvo huecos."""
    # solo las que realmente se imputaron (si no hubo NaN en el crudo, no hay nada que auditar)
    columnas = [c for c in columnas if df_crudo[c].isna().sum() > 0]
    if not columnas:
        print("Ninguna de esas columnas tenía NaN.")
        return
    fig, axes = plt.subplots(len(columnas), 1,
                             figsize=(14, 2.2 * len(columnas)), sharex=True)
    axes = [axes] if len(columnas) == 1 else axes
    for ax, col in zip(axes, columnas):
        imputado = df_crudo[col].isna()
        ax.plot(df_imp['datetime_utc'], df_imp[col], color='lightgray', lw=0.7)
        ax.scatter(df_imp.loc[imputado, 'datetime_utc'], df_imp.loc[imputado, col],
                   color='crimson', s=5)
        ax.set_title(f"{col} · {imputado.sum()} imputados", fontsize=9, loc='left')
    plt.tight_layout()
    plt.show()

In [ ]:
for prefijo in ["tmed_", "tmax_", "velmedia_", "racha_", "prec_", "sol_"]:
    cols = df_copia.columns[df_copia.columns.str.startswith(prefijo)].tolist()
    auditar_imputacion(df, df_copia, cols)

In [ ]:
# ── Marca de completitud (modelo predictivo) ──
# Se calcula AQUÍ, tras crear los lags/rolling, para que sus NaN cuenten.
# (Antes se calculaba antes de los lags → filas con precio_lag_168 = NaN quedaban
#  marcadas True. Recalcular tras los lags también captura los huecos que el
#  .map() deja en los días de cambio de hora / DST.)
# Fuera de las features válidas: target, leakage (precio_portugal + *_real,
# reservados al modelo explicativo) y columnas clave/temporales.
excluir = ["precio_espana", "precio_portugal", "datetime_utc", "fecha"]
excluir += df_copia.columns[df_copia.columns.str.endswith("_real")].tolist()

features_validas = df_copia.columns.drop(excluir)

df_copia["completa_features"] = df_copia[features_validas].notna().all(axis=1)
df_copia["entrenable"]        = df_copia["completa_features"] & df_copia["precio_espana"].notna()

df_copia[["completa_features", "entrenable"]].sum()


In [ ]:

ruta = "../data/processed/tabla_features.parquet"
df_copia.to_parquet(ruta, index=False)

# Verificación rápida de que se guardó bien
comprobar = pd.read_parquet(ruta)
print(comprobar.shape)
print("OK" if comprobar.shape == df_copia.shape else "REVISAR")